In [1]:
import numpy as np
import torch
import transformers

In [2]:
import math

In [5]:
B = 2
T = 4
C = 8

In [6]:
x = torch.randn(B,T,C)

In [8]:
x.shape

torch.Size([2, 4, 8])

In [18]:
Wq = torch.nn.Linear(C,C,bias = False)
Wk = torch.nn.Linear(C,C,bias = False)
Wv = torch.nn.Linear(C,C,bias = False)

In [19]:
q = Wq(x)
k = Wk(x)
v = Wv(x)

In [20]:
q.shape

torch.Size([2, 4, 8])

In [66]:
output = q @ k.transpose(-2,-1) / math.sqrt(C)

In [67]:
mask = torch.tril(torch.ones(T,T))

In [68]:
mask.shape

torch.Size([4, 4])

In [69]:
output.shape

torch.Size([2, 4, 4])

In [70]:
output = output.masked_fill(mask == 0, float("-inf"))

In [71]:
output = torch.softmax(output, dim = -1)

In [72]:
output.shape

torch.Size([2, 4, 4])

In [73]:
v.shape

torch.Size([2, 4, 8])

In [60]:
output = output @ v

In [63]:
output.shape

torch.Size([2, 4, 8])

In [78]:
import torch.nn as nn

In [81]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.query = nn.Linear(d_model, d_model, bias = False)
        self.key = nn.Linear(d_model,d_model, bias = False)
        self.value = nn.Linear(d_model,d_model,bias = False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-2,-1) / (C ** 0.5)
        mask = torch.tril(torch.ones(T,T), device = x.device)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        scores = torch.softmax(scores, dim = -1)
        out = scores @ v

        return out

In [82]:
class CausalMHAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        assert d_model % n_heads == 0
        self.num_heads = n_heads
        self.head_dim = d_model // n_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        H = self.num_heads
        D = self.head_dim

        q = self.query(x).view(B,T,H,D).transpose(1,2)
        k = self.key(x).view(B,T,H,D).transpose(1,2)
        v = self.value(x).view(B,T,H,D).transpose(1,2)

        scores = q @ k.transpose(-2,-1) / (D ** 0.5) # (B,H,T,D) @ (B,H,D,T) -> (B,H,T,T)
        mask = torch.tril(torch.ones(T,T), device = x.device)
        scores = scores.masked_fill(mask == 0, float("-inf"))

        scores = torch.softmax(scores, dim = -1)

        out = scores @ v # (B,H,T,T) @ (B,H,T,D) -> (B,H,T,D)
        out = out.transpose(1,2).reshape(B,T,C)
        
        out = self.out_proj(out)

        return out



In [83]:
class TransformerBlock(nn.Module):
    def __init__(self,d_model,n_heads):
        super().__init__()

        self.attention = CausalMHAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model)
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))

        return x

In [ ]:
class MiniGpt(nn.Module):
    def __init__(self, n_blocks, n_heads, d_model, max_seq_length, vocab_size):
        super().__init__()

        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = nn.Embedding(max_seq_length, d_model)
        self.final_norm = nn.LayerNorm(d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads)
            for _ in range(n_blocks)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape

        token_embeddings = self.embeddings(idx)
        positions = torch.arrange(T, device = idx.device)
        positional_embeddings = self.positional_encoding(positions)

        x = token_embeddings + positional_embeddings
        
        for block in self.blocks:
            x = block(x)
        
        x = self.final_norm(x)
        logits = self.lm_head(x)

        return logits
